## Product performance

Investigates:
- products with high views but low purchase conversion
- products with high revenue
- products with high net margin
- category-level performance
- view → cart → purchase relationships

In [ ]:
# Import Python packages
from snowflake.snowpark.context import get_active_session
import pandas as pd

# Get the current credentials
session = get_active_session()

### Loading product performance

In [ ]:
products = session.table("PRODUCT_ANALYTICS.ANALYTICS.PRODUCT_PERFORMANCE").to_pandas()

products.sort_values(
    "VIEW_TO_PURCHASE_RATE",
    ascending=False
).head(10)

In [ ]:
print(products.groupby("CATEGORY").agg(
    products=("PRODUCT_ID", "count"),
    revenue=("NET_PRODUCT_REVENUE", "sum"),
    units=("UNITS_SOLD", "sum"),
    conversion=("VIEW_TO_PURCHASE_RATE", "mean")
))

### Loading reviews

In [ ]:
reviews = session.table("PRODUCT_ANALYTICS.RAW_EVENTS.REVIEWS").to_pandas()

reviews.head()

In [ ]:
print(reviews["RATING"].value_counts().sort_index())
print(reviews["REVIEW_TEXT"].isna().mean())

In [ ]:
reviews.info()

In [ ]:
reviews.set_index("REVIEW_TIME").resample("ME").size()

### Loading products

In [ ]:
reviews_products = session.table("PRODUCT_ANALYTICS.RAW_EVENTS.PRODUCTS").to_pandas()

reviews_products = reviews.merge(
    products,
    on="PRODUCT_ID",
    how="left"
)

In [ ]:
product_reviews = (
    reviews_products
    .groupby(["PRODUCT_ID", "NAME", "CATEGORY"])
    .agg(
        reviews=("REVIEW_ID", "count"),
        avg_rating=("RATING", "mean")
    )
    .reset_index()
)

In [ ]:
product_reviews[
    product_reviews["reviews"] >= 20
].sort_values(
    "avg_rating",
    ascending=False
).head(10)

### Rating vs. product performance

- Do highly rated products convert better?
- Do poorly rated products have lower repeat purchases?
- Are there products with high traffic + low ratings + low conversion?
- Which categories have the best/worst customer satisfaction?

In [ ]:
product_analysis = product_reviews.merge(
    products,
    on="PRODUCT_ID",
    how="left"
)

In [ ]:
product_analysis.head()

Is customer satisfaction associated with product conversion and revenue?

In [ ]:
print(product_analysis["VIEW_TO_PURCHASE_RATE"].isna().sum())
print(product_analysis["VIEW_TO_PURCHASE_RATE"].notna().sum())

In [ ]:
reviews.info()

### Sentiment analysis

One fo the options to perform a sentiment analysis on the reviews would be iterating through reviews and using Cortex functions if there are available credits:

<code> print(session.sql("SELECT AI_SENTIMENT('This product is excellent and I love it!')").collect()) </code>

Another method for sentiment analysis is through transformers model via HuggingFace like below:

In [ ]:
from snowflake.ml.model.models import huggingface
from snowflake.ml.registry import Registry

# Create the Hugging Face model wrapper
model = huggingface.TransformersPipeline(
    task="text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    top_k=None
)

# Use warehouse for Snowpark operations
session.use_warehouse("COMPUTE_WH")

# Registry
reg = Registry(
    session,
    database_name="PRODUCT_ANALYTICS",
    schema_name="ANALYTICS"
)

# Register model
mv = reg.log_model(
    model=model,
    model_name="roberta",
    version_name="v2"
)

print("Model registered successfully!")
print(mv.show_functions())

Next, we need to deploy the models in Snowsight via Models -> Roberta -> Deploy. After that, we can use the model for inference.

In [ ]:
session.use_database("PRODUCT_ANALYTICS")

mp = reg.get_model("PRODUCT_ANALYTICS.ANALYTICS.ROBERTA").version("V2")

input_df = reviews[["REVIEW_TEXT"]].rename(
    columns={"REVIEW_TEXT": "text"}
)

output_df = mp.run(
    input_df,
    params={"top_k": None, "function_to_apply": None},
    function_name="__call__",
    service_name="PRODUCT_ANALYTICS.ANALYTICS.ROBERTA_V2_SERVICE"
)

print(output_df.head())


However, in this case, the service is not able to run and throws this error:

<code>snowflake.ml._internal.exceptions.exceptions.SnowflakeMLException: ValueError('(2110) Data does not have the same number of features as signature. Signature requires 2 features, but have 1 in input data.')</code>

But the printed functions upon model creation show that the model only needs 1 parameter as an input. Could be an issue on a call method for the inference service.

In [ ]:
reviews["SENTIMENT"] = [
    r[0]["label"] for r in output_df
]

reviews["SENTIMENT"].value_counts()

In [ ]:
reviews.groupby("SENTIMENT")["RATING"].agg(
    ["count", "mean"]
)

### TF-IDF weighted document-term matrix

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

negative = reviews.loc[
    reviews["RATING"] <= 2,
    "REVIEW_TEXT"
].dropna()

vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=3,
    max_features=50
)

X = vectorizer.fit_transform(negative)

terms = pd.DataFrame({
    "term": vectorizer.get_feature_names_out(),
    "score": X.mean(axis=0).A1
}).sort_values(
    "score",
    ascending=False
)

terms.head(20)